# Governed Compressed Market Day
## A transparent five-minute Colab reconstruction of the Obsidian-vault demonstration

**Purpose.** This notebook reproduces the classroom exercise in which five one-minute market states are accumulated in an Obsidian vault and transformed into a path-dependent end-of-day narrative.

The notebook makes the entire process inspectable:

1. create a Drive-backed Obsidian vault;
2. register sources and limitations;
3. establish five market benchmarks;
4. generate five explicitly **SIMULATED** intraday states;
5. make every report consult its predecessors;
6. construct an end-of-day narrative from the full trajectory;
7. produce a dataset, lineage graph, validation report, cryptographic hashes, and audit manifest.

> **Governance boundary:** the five intraday states are pedagogical scenario data, not live Yahoo Finance quotations. The notebook never silently converts simulated data into observed facts.

## 0. What the original Work demonstration did

The original request asked for a report every minute using Yahoo Finance and prior reports. Two practical constraints were discovered: Work automations do not support one-minute recurrence, and Yahoo restricted direct retrieval. The implementation therefore compressed one market day into five classroom minutes and separated:

- **VERIFIED_BASELINE** — externally supported context recorded in the source register;
- **SIMULATED** — the five pedagogical intraday market states;
- **DERIVED** — calculations made from those states;
- **INTERPRETATION** — narrative conclusions;
- **UNAVAILABLE** — observations that could not be verified.

This is the central lesson: transparency reveals the workflow; governance controls what the workflow is allowed to claim.

In [1]:
# 1. Mount Google Drive (Colab only)
from pathlib import Path
import os, json, csv, hashlib, textwrap
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local runtime detected; Drive mount skipped.')

Mounted at /content/drive


In [2]:
# 2. Configuration — change only VAULT_ROOT if desired
if IN_COLAB:
    VAULT_ROOT = Path('/content/drive/MyDrive/ESSENTIAL AUTONOMOUS WOKFLOWS/THE FINANCIAL REPORTER')
else:
    VAULT_ROOT = Path.cwd() / 'Market_Day_Obsidian_Vault'

RUN_ID = 'compressed-market-day-2026-07-22'
RUN_DATE = '2026-07-22'
TIMEZONE = 'America/Mexico_City'
SCENARIO_STATUS = 'SIMULATED'

DIRS = {
    'governance': VAULT_ROOT / '00_Governance',
    'reports': VAULT_ROOT / '01_Minute_Reports',
    'narrative': VAULT_ROOT / '02_End_of_Day',
    'data': VAULT_ROOT / '03_Data',
    'audit': VAULT_ROOT / '04_Audit',
}
for p in DIRS.values(): p.mkdir(parents=True, exist_ok=True)
print('Vault:', VAULT_ROOT)

Vault: /content/drive/MyDrive/ESSENTIAL AUTONOMOUS WOKFLOWS/THE FINANCIAL REPORTER


## 1. Instruments and source register

The five benchmarks match the original demonstration. The source register records where evidence would come from and what role each source played. URLs are references—not a claim that this notebook successfully scraped them.

In [3]:
INDEXES = [
    {'symbol': '^GSPC', 'name': 'S&P 500', 'region': 'United States'},
    {'symbol': '^DJI',  'name': 'Dow Jones Industrial Average', 'region': 'United States'},
    {'symbol': '^IXIC', 'name': 'Nasdaq Composite', 'region': 'United States'},
    {'symbol': '^FTSE', 'name': 'FTSE 100', 'region': 'United Kingdom'},
    {'symbol': '^N225', 'name': 'Nikkei 225', 'region': 'Japan'},
]

SOURCES = [
    {'source_id':'SRC-001','publisher':'Yahoo Finance','url':'https://finance.yahoo.com/markets/world-indices/',
     'role':'Intended market-index baseline','status':'ACCESS_RESTRICTED_DURING_DEMO'},
    {'source_id':'SRC-002','publisher':'Reuters','url':'https://www.reuters.com/markets/us/global-markets-technicals-graphic-2026-07-22/',
     'role':'Contemporary market context','status':'REFERENCE_REGISTERED'},
    {'source_id':'SRC-003','publisher':'Associated Press','url':'https://apnews.com/article/ccf404ea3258636974afa17f714db8e8',
     'role':'Contemporary market context','status':'REFERENCE_REGISTERED'},
]
INDEXES, SOURCES

([{'symbol': '^GSPC', 'name': 'S&P 500', 'region': 'United States'},
  {'symbol': '^DJI',
   'name': 'Dow Jones Industrial Average',
   'region': 'United States'},
  {'symbol': '^IXIC', 'name': 'Nasdaq Composite', 'region': 'United States'},
  {'symbol': '^FTSE', 'name': 'FTSE 100', 'region': 'United Kingdom'},
  {'symbol': '^N225', 'name': 'Nikkei 225', 'region': 'Japan'}],
 [{'source_id': 'SRC-001',
   'publisher': 'Yahoo Finance',
   'url': 'https://finance.yahoo.com/markets/world-indices/',
   'role': 'Intended market-index baseline',
   'status': 'ACCESS_RESTRICTED_DURING_DEMO'},
  {'source_id': 'SRC-002',
   'publisher': 'Reuters',
   'url': 'https://www.reuters.com/markets/us/global-markets-technicals-graphic-2026-07-22/',
   'role': 'Contemporary market context',
   'status': 'REFERENCE_REGISTERED'},
  {'source_id': 'SRC-003',
   'publisher': 'Associated Press',
   'url': 'https://apnews.com/article/ccf404ea3258636974afa17f714db8e8',
   'role': 'Contemporary market context',
  

In [4]:
# 3. Controlled classroom scenario: percent change from baseline
# These values are illustrative and are never represented as live quotations.
STATES = [
 {'minute':1,'time':'09:30','label':'Opening shock',
  'moves':{'^GSPC':-0.45,'^DJI':-0.30,'^IXIC':-0.70,'^FTSE':-0.20,'^N225':-0.55},
  'interpretation':'A broad risk-off opening, led by technology and Japan.'},
 {'minute':2,'time':'09:31','label':'Early selloff',
  'moves':{'^GSPC':-0.82,'^DJI':-0.58,'^IXIC':-1.18,'^FTSE':-0.42,'^N225':-0.90},
  'interpretation':'Selling broadens; the downside move accelerates and contagion risk rises.'},
 {'minute':3,'time':'09:32','label':'Reversal',
  'moves':{'^GSPC':-0.28,'^DJI':-0.22,'^IXIC':-0.38,'^FTSE':-0.18,'^N225':-0.62},
  'interpretation':'US indices recover sharply, but Japan remains a laggard.'},
 {'minute':4,'time':'09:33','label':'Market divergence',
  'moves':{'^GSPC':0.05,'^DJI':0.18,'^IXIC':-0.12,'^FTSE':0.10,'^N225':-0.48},
  'interpretation':'The tape turns selective: value-oriented benchmarks stabilize before technology.'},
 {'minute':5,'time':'09:34','label':'Close',
  'moves':{'^GSPC':0.22,'^DJI':0.35,'^IXIC':0.08,'^FTSE':0.16,'^N225':-0.30},
  'interpretation':'Most benchmarks finish positive, while Japan retains a material loss.'},
]
assert len(STATES) == 5 and all(len(s['moves']) == 5 for s in STATES)
STATES

[{'minute': 1,
  'time': '09:30',
  'label': 'Opening shock',
  'moves': {'^GSPC': -0.45,
   '^DJI': -0.3,
   '^IXIC': -0.7,
   '^FTSE': -0.2,
   '^N225': -0.55},
  'interpretation': 'A broad risk-off opening, led by technology and Japan.'},
 {'minute': 2,
  'time': '09:31',
  'label': 'Early selloff',
  'moves': {'^GSPC': -0.82,
   '^DJI': -0.58,
   '^IXIC': -1.18,
   '^FTSE': -0.42,
   '^N225': -0.9},
  'interpretation': 'Selling broadens; the downside move accelerates and contagion risk rises.'},
 {'minute': 3,
  'time': '09:32',
  'label': 'Reversal',
  'moves': {'^GSPC': -0.28,
   '^DJI': -0.22,
   '^IXIC': -0.38,
   '^FTSE': -0.18,
   '^N225': -0.62},
  'interpretation': 'US indices recover sharply, but Japan remains a laggard.'},
 {'minute': 4,
  'time': '09:33',
  'label': 'Market divergence',
  'moves': {'^GSPC': 0.05,
   '^DJI': 0.18,
   '^IXIC': -0.12,
   '^FTSE': 0.1,
   '^N225': -0.48},
  'interpretation': 'The tape turns selective: value-oriented benchmarks stabilize befo

## 2. Governance gates

Each report passes four controls before it is written:

| Gate | Test | Failure consequence |
|---|---|---|
| G1 Provenance | Every value has a classification | Stop |
| G2 Simulation disclosure | Scenario notes visibly say `SIMULATED` | Stop |
| G3 Memory lineage | Minute *n* cites all prior minute notes | Stop |
| G4 Completeness | All five indexes are present | Stop |

The notebook also uses file hashes after creation. Those hashes do not prove that the content is true; they prove which exact bytes belonged to this run.

In [5]:
def slug(n): return f"Minute_{n:02d}"

def predecessors(minute):
    return [f'[[{slug(i)}]]' for i in range(1, minute)]

def validate_state(state):
    errors = []
    if SCENARIO_STATUS != 'SIMULATED': errors.append('Scenario is not explicitly SIMULATED')
    if set(state['moves']) != {x['symbol'] for x in INDEXES}: errors.append('Index universe incomplete')
    if state['minute'] > 1 and len(predecessors(state['minute'])) != state['minute'] - 1:
        errors.append('Predecessor chain incomplete')
    if errors: raise ValueError('; '.join(errors))
    return True

for state in STATES: validate_state(state)
print('All pre-write governance gates passed.')

All pre-write governance gates passed.


In [6]:
# 4. Write governance and methodology notes
methodology = f"""---
title: Governance and Methodology
run_id: {RUN_ID}
date: {RUN_DATE}
status: GOVERNED_CLASSROOM_SIMULATION
---

# Governance and Methodology

## Objective
Create five sequential market reports in five classroom minutes, with every report using the vault's earlier reports, then synthesize the full path into an end-of-day narrative.

## Data classifications
- `VERIFIED_BASELINE`: contextual fact with a registered external source.
- `SIMULATED`: pedagogical state created for this compressed exercise.
- `DERIVED`: deterministic calculation from registered inputs.
- `INTERPRETATION`: analytical narrative.
- `UNAVAILABLE`: requested observation not verified.

## Constraint disclosure
One-minute Work recurrence was unavailable and Yahoo restricted direct reads during the demonstration. The five states are therefore explicitly simulated. No simulated value is represented as a live quote.

## Control gates
G1 provenance; G2 simulation disclosure; G3 predecessor lineage; G4 universe completeness.
"""
(DIRS['governance'] / 'Governance_and_Methodology.md').write_text(methodology, encoding='utf-8')

source_note = '# Source Register\n\n' + '\n'.join(
    f"- **{s['source_id']} — {s['publisher']}**: [{s['url']}]({s['url']})  \n  Role: {s['role']}  \n  Status: `{s['status']}`" for s in SOURCES)
(DIRS['governance'] / 'Source_Register.md').write_text(source_note, encoding='utf-8')
print('Governance notes written.')

Governance notes written.


## 3. Sequential report generation

The key design feature is **path dependence**. Minute 5 does not merely see the latest vector; it inherits Minutes 1–4. Each Markdown file has Obsidian `[[wikilinks]]` to every predecessor and contains a compact “memory used” summary.

In [7]:
def market_table(state):
    rows = ['| Index | Symbol | Scenario move (%) | Classification |',
            '|---|---:|---:|---|']
    by_symbol = {x['symbol']: x for x in INDEXES}
    for symbol, move in state['moves'].items():
        rows.append(f"| {by_symbol[symbol]['name']} | `{symbol}` | {move:+.2f}% | SIMULATED |")
    return '\n'.join(rows)

def trajectory_summary(up_to):
    selected = STATES[:up_to]
    worst = min((v, s['minute'], sym) for s in selected for sym, v in s['moves'].items())
    latest = selected[-1]
    positives = sum(v > 0 for v in latest['moves'].values())
    return (f"The history now contains {up_to} state(s). The deepest observed scenario move so far "
            f"is {worst[0]:+.2f}% in {worst[2]} at Minute {worst[1]}. "
            f"At the current state, {positives} of 5 benchmarks are positive.")

created_reports = []
for state in STATES:
    minute = state['minute']
    prior = predecessors(minute)
    prior_text = ', '.join(prior) if prior else 'None — this is the first report.'
    note = f"""---
title: Minute {minute:02d} — {state['label']}
run_id: {RUN_ID}
minute: {minute}
scenario_time: {state['time']}
classification: SIMULATED
prior_reports_consulted: {minute-1}
---

# Minute {minute:02d} — {state['label']}

> [!warning] SIMULATED CLASSROOM STATE
> These are pedagogical scenario values—not live or historical Yahoo Finance quotations.

## Memory consulted
{prior_text}

## Market state
{market_table(state)}

## Cumulative memory
{trajectory_summary(minute)}

## Interpretation
**INTERPRETATION:** {state['interpretation']}

## Lineage
- Method: [[Governance_and_Methodology]]
- Sources: [[Source_Register]]
- Predecessor count: {minute-1}
- Input classification: `SIMULATED`
"""
    path = DIRS['reports'] / f'{slug(minute)}.md'
    path.write_text(note, encoding='utf-8')
    created_reports.append(path)
    print('Created', path.name, '| consulted', minute-1, 'prior report(s)')

Created Minute_01.md | consulted 0 prior report(s)
Created Minute_02.md | consulted 1 prior report(s)
Created Minute_03.md | consulted 2 prior report(s)
Created Minute_04.md | consulted 3 prior report(s)
Created Minute_05.md | consulted 4 prior report(s)


In [17]:
# 5. Create the structured long-form CSV dataset
csv_path = DIRS['data'] / 'market_states.csv'
fields = ['run_id','date','minute','scenario_time','phase','symbol','index_name','region','move_pct','classification']
index_map = {x['symbol']:x for x in INDEXES}
with csv_path.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
    for s in STATES:
        for symbol, move in s['moves'].items():
            x = index_map[symbol]
            w.writerow({'run_id':RUN_ID,'date':RUN_DATE,'minute':s['minute'],'scenario_time':s['time'],
                        'phase':s['label'],'symbol':symbol,'index_name':x['name'],'region':x['region'],
                        'move_pct':move,'classification':'SIMULATED'})

# Add verification step
if csv_path.exists():
    print(f"Successfully created {csv_path.name} at {csv_path}. Total rows expected: {5*5}. File size: {csv_path.stat().st_size} bytes")
else:
    print(f"Error: {csv_path.name} was not created.")

Successfully created market_states.csv at /content/drive/MyDrive/ESSENTIAL AUTONOMOUS WOKFLOWS/THE FINANCIAL REPORTER/03_Data/market_states.csv. Total rows expected: 25. File size: 2936 bytes


## 4. Path-dependent narrative synthesis

A closing snapshot alone would say “four markets rose and one fell.” The accumulated vault supports a more interesting account: opening shock → expanding selloff → reversal → divergence → differentiated close.

In [18]:
# 6. Deterministically derive trajectory metrics
metrics = {}
for x in INDEXES:
    sym = x['symbol']
    path = [s['moves'][sym] for s in STATES]
    metrics[sym] = {
        'open': path[0], 'trough': min(path), 'trough_minute': path.index(min(path))+1,
        'close': path[-1], 'recovery_from_trough': path[-1]-min(path),
        'range': max(path)-min(path)
    }
metrics

{'^GSPC': {'open': -0.45,
  'trough': -0.82,
  'trough_minute': 2,
  'close': 0.22,
  'recovery_from_trough': 1.04,
  'range': 1.04},
 '^DJI': {'open': -0.3,
  'trough': -0.58,
  'trough_minute': 2,
  'close': 0.35,
  'recovery_from_trough': 0.9299999999999999,
  'range': 0.9299999999999999},
 '^IXIC': {'open': -0.7,
  'trough': -1.18,
  'trough_minute': 2,
  'close': 0.08,
  'recovery_from_trough': 1.26,
  'range': 1.26},
 '^FTSE': {'open': -0.2,
  'trough': -0.42,
  'trough_minute': 2,
  'close': 0.16,
  'recovery_from_trough': 0.58,
  'range': 0.58},
 '^N225': {'open': -0.55,
  'trough': -0.9,
  'trough_minute': 2,
  'close': -0.3,
  'recovery_from_trough': 0.6000000000000001,
  'range': 0.6000000000000001}}

In [19]:
# 7. Write the end-of-day narrative
index_map = {x['symbol']:x for x in INDEXES}
close_positive = [index_map[s]['name'] for s,m in metrics.items() if m['close'] > 0]
laggards = [index_map[s]['name'] for s,m in metrics.items() if m['close'] <= 0]
best_recovery_symbol = max(metrics, key=lambda s: metrics[s]['recovery_from_trough'])

narrative = f"""---
title: End-of-Day Narrative
run_id: {RUN_ID}
date: {RUN_DATE}
classification: INTERPRETATION_FROM_SIMULATED_DATA
---

# End-of-Day Narrative

> [!warning] GOVERNED CLASSROOM SIMULATION
> This narrative is derived from five explicitly simulated market states. It is not a report of actual trading on {RUN_DATE}.

## Reports consulted
{', '.join(f'[[{slug(i)}]]' for i in range(1,6))}

## The day's arc

The compressed day began with a synchronized risk-off shock. The Nasdaq Composite and Nikkei 225 led the opening losses, suggesting that the initial pressure was concentrated in technology-sensitive and international risk exposures. The second minute changed the character of the episode: losses deepened across all five benchmarks, transforming an opening wobble into a broad selloff and raising the possibility of contagion.

That possibility did not materialize uniformly. Minute 3 introduced a decisive US reversal. The recovery was strongest in **{index_map[best_recovery_symbol]['name']}**, which regained **{metrics[best_recovery_symbol]['recovery_from_trough']:.2f} percentage points** from trough to close. By Minute 4, the market was no longer moving as a single block: the Dow and FTSE had crossed into positive territory while the Nasdaq remained slightly negative and the Nikkei lagged.

The final state therefore concealed a much richer history than its endpoint suggested. **{', '.join(close_positive)}** closed positive in the scenario, while **{', '.join(laggards)}** remained negative. The governing narrative is not simply “markets rose.” It is: **shock → contagion risk → reversal → sector and regional divergence → differentiated close**.

## Why vault memory mattered

Without Minutes 1–4, the close would reveal direction but not stress, timing, recovery, or leadership rotation. The vault converts isolated snapshots into a trajectory and makes the evidence chain inspectable.

## Classification
- Scenario values: `SIMULATED`
- Metrics: `DERIVED`
- Prose: `INTERPRETATION`
"""
narrative_path = DIRS['narrative'] / 'End_of_Day_Narrative.md'
narrative_path.write_text(narrative, encoding='utf-8')
print(narrative_path)

/content/drive/MyDrive/ESSENTIAL AUTONOMOUS WOKFLOWS/THE FINANCIAL REPORTER/02_End_of_Day/End_of_Day_Narrative.md


In [20]:
# 8. Write an Obsidian navigation index and lineage diagram
home = f"""# Market Day Vault

## Governance
- [[Governance_and_Methodology]]
- [[Source_Register]]

## Sequential reports
""" + '\n'.join(f'- [[{slug(i)}]]' for i in range(1,6)) + """

## Synthesis
- [[End_of_Day_Narrative]]

## Lineage
```mermaid
flowchart LR
  M1[Minute 1] --> M2[Minute 2]
  M2 --> M3[Minute 3]
  M3 --> M4[Minute 4]
  M4 --> M5[Minute 5]
  M5 --> EOD[End-of-Day Narrative]
```
"""
(VAULT_ROOT / 'HOME.md').write_text(home, encoding='utf-8')
print('HOME.md written.')

HOME.md written.


## 5. Audit bundle

The manifest records run identity, file inventory, classifications, control results, and SHA-256 hashes. Validation is performed against the generated files—not merely against in-memory objects.

In [21]:
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(65536), b''): h.update(chunk)
    return h.hexdigest()

def all_artifacts():
    return sorted(p for p in VAULT_ROOT.rglob('*') if p.is_file())

validation = {'status':'PASS','checks':[]}
for i, path in enumerate(created_reports, start=1):
    text = path.read_text(encoding='utf-8')
    tests = {
        'simulation_disclosed': 'SIMULATED CLASSROOM STATE' in text,
        'classification_present': 'classification: SIMULATED' in text,
        'all_predecessors_linked': all(f'[[{slug(j)}]]' in text for j in range(1,i)),
        'five_indexes_present': all(x['name'] in text for x in INDEXES),
    }
    validation['checks'].append({'file':path.name, **tests})
    if not all(tests.values()): validation['status'] = 'FAIL'

if validation['status'] != 'PASS': raise RuntimeError('Generated vault failed governance validation')
(DIRS['audit'] / 'validation_report.json').write_text(json.dumps(validation, indent=2), encoding='utf-8')
print(json.dumps(validation, indent=2))

{
  "status": "PASS",
  "checks": [
    {
      "file": "Minute_01.md",
      "simulation_disclosed": true,
      "classification_present": true,
      "all_predecessors_linked": true,
      "five_indexes_present": true
    },
    {
      "file": "Minute_02.md",
      "simulation_disclosed": true,
      "classification_present": true,
      "all_predecessors_linked": true,
      "five_indexes_present": true
    },
    {
      "file": "Minute_03.md",
      "simulation_disclosed": true,
      "classification_present": true,
      "all_predecessors_linked": true,
      "five_indexes_present": true
    },
    {
      "file": "Minute_04.md",
      "simulation_disclosed": true,
      "classification_present": true,
      "all_predecessors_linked": true,
      "five_indexes_present": true
    },
    {
      "file": "Minute_05.md",
      "simulation_disclosed": true,
      "classification_present": true,
      "all_predecessors_linked": true,
      "five_indexes_present": true
    }
  ]
}


In [22]:
# 9. Final manifest (hashes computed after all other files exist)
artifacts_before_manifest = all_artifacts()
manifest = {
    'schema_version':'1.0',
    'run_id':RUN_ID,
    'run_date':RUN_DATE,
    'timezone':TIMEZONE,
    'purpose':'Five-minute governed classroom simulation of a market day',
    'scenario_status':'SIMULATED',
    'automation_status':'NOT_ACTIVATED_ONE_MINUTE_RECURRENCE_UNSUPPORTED',
    'source_retrieval_limitation':'Yahoo direct retrieval was restricted during the original demonstration',
    'universe':INDEXES,
    'sources':SOURCES,
    'gates':{'G1_provenance':'PASS','G2_simulation_disclosure':'PASS',
             'G3_memory_lineage':'PASS','G4_completeness':'PASS'},
    'files':[{'path':str(p.relative_to(VAULT_ROOT)),'sha256':sha256(p),'bytes':p.stat().st_size}
             for p in artifacts_before_manifest],
}
manifest_path = DIRS['audit'] / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Manifest:', manifest_path)
print('Artifacts registered:', len(manifest['files']))

Manifest: /content/drive/MyDrive/ESSENTIAL AUTONOMOUS WOKFLOWS/THE FINANCIAL REPORTER/04_Audit/run_manifest.json
Artifacts registered: 12


In [23]:
# 10. Final vault verification from disk
files = all_artifacts()
for p in files:
    print(f'{p.relative_to(VAULT_ROOT)!s:55} {p.stat().st_size:>8,} bytes')

required = {
 'HOME.md','00_Governance/Governance_and_Methodology.md','00_Governance/Source_Register.md',
 '01_Minute_Reports/Minute_01.md','01_Minute_Reports/Minute_02.md','01_Minute_Reports/Minute_03.md',
 '01_Minute_Reports/Minute_04.md','01_Minute_Reports/Minute_05.md',
 '02_End_of_Day/End_of_Day_Narrative.md','03_Data/market_states.csv',
 '04_Audit/validation_report.json','04_Audit/run_manifest.json'}
actual = {str(p.relative_to(VAULT_ROOT)) for p in files}
assert required <= actual
print(f'\nPASS — complete governed vault contains {len(files)} files.')

00_Governance/Governance_and_Methodology.md                1,029 bytes
00_Governance/Source_Register.md                             734 bytes
01_Minute_Reports/Minute_01.md                             1,151 bytes
01_Minute_Reports/Minute_02.md                             1,149 bytes
01_Minute_Reports/Minute_03.md                             1,137 bytes
01_Minute_Reports/Minute_04.md                             1,194 bytes
01_Minute_Reports/Minute_05.md                             1,174 bytes
02_End_of_Day/End_of_Day_Narrative.md                      2,013 bytes
03_Data/market_states.csv                                  2,936 bytes
04_Audit/run_manifest.json                                 3,843 bytes
04_Audit/validation_report.json                              995 bytes
HOME.md                                                      406 bytes

PASS — complete governed vault contains 12 files.


## 6. Classroom discussion

1. **Why is the closing snapshot insufficient?** It omits the depth of the early selloff, the reversal, and changing leadership.
2. **What did Obsidian contribute?** Durable, human-readable memory and explicit links—not market-data retrieval itself.
3. **What does the manifest prove?** Which artifacts belonged to the run and whether required controls passed; it does not independently prove market truth.
4. **Why not disguise the scenario as live data?** Because pedagogical convenience cannot override provenance.
5. **How would production differ?** Use a licensed market-data API, authenticated timestamps, retries, trading-calendar awareness, orchestration at the required frequency, immutable raw observations, and human approval for externally distributed narratives.

### Production extension (not executed here)

Replace `STATES` with a licensed provider adapter that returns values plus source timestamp, retrieval timestamp, vendor, instrument identifier, and response hash. Keep the same gates, lineage, vault-writing, narrative, and manifest layers. This separation is precisely what makes the architecture governable.